In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [10]:
df = pd.read_csv("loan_dataset.csv", low_memory=False)

# Reduce size for performance
df = df.sample(n=30000, random_state=42)

print("Shape:", df.shape)
df.head()

Shape: (30000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
392949,39651438,NaN,32000.0,32000.0,32000.0,60 months,10.49,687.65,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1273506,16411620,NaN,9600.0,9600.0,9600.0,36 months,12.99,323.42,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
324024,45122316,NaN,4000.0,4000.0,4000.0,36 months,6.68,122.93,A,A3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2066630,125356772,NaN,6025.0,6025.0,6025.0,36 months,10.91,197.00,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
477199,128490686,NaN,25000.0,25000.0,25000.0,60 months,26.30,752.96,E,E5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

df['loan_status'] = df['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})

df['loan_status'].value_counts()

In [ ]:
# Drop columns with >50% missing
df = df.dropna(thresh=len(df)*0.5, axis=1)

# Fill numeric
num_cols = df.select_dtypes(include=['int64','float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

print("After cleaning:", df.shape)

In [ ]:
# FICO combine
if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
    df['fico_score'] = (df['fico_range_low'] + df['fico_range_high']) / 2
    df.drop(['fico_range_low','fico_range_high'], axis=1, inplace=True)

# Term
if 'term' in df.columns:
    df['term'] = df['term'].str.extract(r'(\d+)').astype(float)

# Employment length
if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].str.replace('< 1 year','0')
    df['emp_length'] = df['emp_length'].str.replace('10+ years','10')
    df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)

df.head()

In [ ]:
df.drop(columns=['id','member_id','url','emp_title','title','zip_code','addr_state'],
        inplace=True, errors='ignore')

print(df.shape)

In [ ]:
sns.histplot(df['loan_amnt'], bins=20)
plt.title("Loan Amount Distribution")
plt.show()

sns.boxplot(x=df['annual_inc'])
plt.title("Annual Income")
plt.show()

In [ ]:
df = pd.get_dummies(df, drop_first=True)

df = df.fillna(df.median(numeric_only=True))

print(df.shape)

In [ ]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

In [ ]:
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

print("After PCA:", X_pca.shape)